In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,roc_auc_score,confusion_matrix,classification_report

### Load Model Input Data

In [7]:
df=pd.read_csv("../Model_Input_Data/Model_Input_Data.csv")
print(df.shape)
df.head()

(5410, 36)


,Provider,PotentialFraud,IP_Claim_Count,IP_Benf_Count,Avg_IP_InscClaimAmtReimbursed,Avg_IP_DeductibleAmtPaid,Avg_IP_Number_of_Days_in_Hospital,Avg_IP_Number_of_Days_in_Hospital.1,Avg_IP_Number_of_Days_in_Hospital.2,Avg_IP_Number_of_Days_in_Hospital.3,...,IschemicHeart_Count,Osteoporosis_Count,RheumatoidArthritis_Count,Stroke_Count,Avg_PartA_Months,Avg_PartB_Months_Mode,Avg_IP_Reimbursement,Avg_IP_Deductible,Avg_OP_Reimbursement,Avg_OP_Deductible
0,PRV51001,0,5,5,19400.000000,1068.0,0.0,0.0,6.0,0.0,...,2,18,16,19,12,12,18047.916667,890.000000,2537.500000,474.916667
1,PRV51003,1,62,53,9241.935484,1068.0,3.0,3.0,9.0,1.0,...,18,89,85,108,12,12,6814.017094,822.632479,2490.598291,664.529915
2,PRV51004,0,0,0,0.000000,0.0,0.0,0.0,0.0,0.0,...,40,95,97,122,12,12,4596.739130,454.144928,2095.144928,600.869565
3,PRV51005,1,0,0,0.000000,0.0,0.0,0.0,0.0,0.0,...,148,353,368,456,12,12,3717.232323,398.698990,1798.808081,475.965657
4,PRV51007,0,3,3,6333.333333,1068.0,4.0,4.0,6.0,0.0,...,18,41,42,49,12,12,3109.655172,423.517241,1497.241379,430.689655


In [22]:
df.PotentialFraud.value_counts()

0    4904
1     506
Name: PotentialFraud, dtype: int64

Here The data was Imbalanced.

### Checking Correlation Coefficent

- build a correlation matrix.
- identify Higly correlated variables.

In [23]:
corr_matrix=df.iloc[:,1:].corr()
corr_matrix

,PotentialFraud,IP_Claim_Count,IP_Benf_Count,Avg_IP_InscClaimAmtReimbursed,Avg_IP_DeductibleAmtPaid,Avg_IP_Number_of_Days_in_Hospital,Avg_IP_Number_of_Days_in_Hospital.1,Avg_IP_Number_of_Days_in_Hospital.2,Avg_IP_Number_of_Days_in_Hospital.3,OP_Claim_Count,...,IschemicHeart_Count,Osteoporosis_Count,RheumatoidArthritis_Count,Stroke_Count,Avg_PartA_Months,Avg_PartB_Months_Mode,Avg_IP_Reimbursement,Avg_IP_Deductible,Avg_OP_Reimbursement,Avg_OP_Deductible
PotentialFraud,1.000000,0.525393,0.522256,0.307556,0.319855,0.156278,0.157271,0.332043,0.203341,0.335803,...,0.369433,0.390565,0.388635,0.389683,0.009770,0.008369,0.127398,0.119897,-0.015204,-0.031547
IP_Claim_Count,0.525393,1.000000,0.997492,0.327985,0.398766,0.223728,0.224221,0.418265,0.228004,0.208758,...,0.325412,0.383660,0.379031,0.382209,0.009087,0.007868,0.165848,0.183794,-0.023449,-0.028603
IP_Benf_Count,0.522256,0.997492,1.000000,0.336416,0.409007,0.229987,0.230475,0.428677,0.231378,0.208244,...,0.328001,0.387186,0.382511,0.386015,0.009265,0.008031,0.170824,0.189412,-0.023422,-0.028680
Avg_IP_InscClaimAmtReimbursed,0.307556,0.327985,0.336416,1.000000,0.839577,0.672207,0.671091,0.827822,0.578765,0.022452,...,0.057874,0.079036,0.077457,0.078331,0.005905,0.010255,0.478656,0.411210,-0.007450,-0.026045
Avg_IP_DeductibleAmtPaid,0.319855,0.398766,0.409007,0.839577,1.000000,0.661765,0.662139,0.976286,0.566851,0.028770,...,0.074164,0.099966,0.097892,0.099275,-0.000936,0.003250,0.396898,0.433707,-0.029665,-0.042322
Avg_IP_Number_of_Days_in_Hospital,0.156278,0.223728,0.229987,0.672207,0.661765,1.000000,0.999488,0.664418,0.457804,0.004660,...,0.027244,0.041171,0.039813,0.040823,0.005033,0.002230,0.330939,0.321009,-0.029046,-0.028366
Avg_IP_Number_of_Days_in_Hospital.1,0.157271,0.224221,0.230475,0.671091,0.662139,0.999488,1.000000,0.665178,0.457536,0.004684,...,0.027227,0.041213,0.039873,0.040851,0.005025,0.002220,0.329952,0.321355,-0.028347,-0.028003
Avg_IP_Number_of_Days_in_Hospital.2,0.332043,0.418265,0.428677,0.827822,0.976286,0.664418,0.665178,1.000000,0.564605,0.035079,...,0.083536,0.110669,0.108475,0.109847,-0.002132,0.002353,0.395796,0.432281,-0.033373,-0.042564
Avg_IP_Number_of_Days_in_Hospital.3,0.203341,0.228004,0.231378,0.578765,0.566851,0.457804,0.457536,0.564605,1.000000,0.026307,...,0.049285,0.064725,0.063202,0.063827,-0.000450,0.011889,0.268412,0.269336,0.014888,0.008427
OP_Claim_Count,0.335803,0.208758,0.208244,0.022452,0.028770,0.004660,0.004684,0.035079,0.026307,1.000000,...,0.955746,0.933761,0.935031,0.931964,0.010603,0.009193,-0.088130,-0.093677,-0.044329,-0.047724


In [24]:
check=corr_matrix['IP_Claim_Count'].apply(lambda x: 0.8>= x<1)
check



PotentialFraud                          True
IP_Claim_Count                         False
IP_Benf_Count                          False
Avg_IP_InscClaimAmtReimbursed           True
Avg_IP_DeductibleAmtPaid                True
Avg_IP_Number_of_Days_in_Hospital       True
Avg_IP_Number_of_Days_in_Hospital.1     True
Avg_IP_Number_of_Days_in_Hospital.2     True
Avg_IP_Number_of_Days_in_Hospital.3     True
OP_Claim_Count                          True
OP_Benf_Count                           True
Avg_OP_InscClaimAmtReimbursed           True
Avg_OP_DeductibleAmtPaid                True
Avg_OP_Number_of_Days_in_Hospital       True
Avg_OP_Number_of_Days_in_Hospital.1     True
Avg_OP_Number_of_Days_in_Hospital.2     True
Total_Beneficiaries                     True
RenalDisease_Count                      True
Alzheimer_Count                         True
HeartFailure_Count                      True
KidneyDisease_Count                     True
Cancer_Count                            True
Pulmonary_

In [25]:
cols = list(corr_matrix.columns)

check = corr_matrix.apply(lambda row :  [row[c] if 0.8<=abs(row[c])<1 else None for c in cols ])
check


,PotentialFraud,IP_Claim_Count,IP_Benf_Count,Avg_IP_InscClaimAmtReimbursed,Avg_IP_DeductibleAmtPaid,Avg_IP_Number_of_Days_in_Hospital,Avg_IP_Number_of_Days_in_Hospital.1,Avg_IP_Number_of_Days_in_Hospital.2,Avg_IP_Number_of_Days_in_Hospital.3,OP_Claim_Count,...,IschemicHeart_Count,Osteoporosis_Count,RheumatoidArthritis_Count,Stroke_Count,Avg_PartA_Months,Avg_PartB_Months_Mode,Avg_IP_Reimbursement,Avg_IP_Deductible,Avg_OP_Reimbursement,Avg_OP_Deductible
PotentialFraud,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,...,NaN,NaN,NaN,NaN,None,None,None,None,NaN,NaN
IP_Claim_Count,None,NaN,0.997492,NaN,NaN,NaN,NaN,NaN,None,NaN,...,NaN,NaN,NaN,NaN,None,None,None,None,NaN,NaN
IP_Benf_Count,None,0.997492,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,...,NaN,NaN,NaN,NaN,None,None,None,None,NaN,NaN
Avg_IP_InscClaimAmtReimbursed,None,NaN,NaN,NaN,0.839577,NaN,NaN,0.827822,None,NaN,...,NaN,NaN,NaN,NaN,None,None,None,None,NaN,NaN
Avg_IP_DeductibleAmtPaid,None,NaN,NaN,0.839577,NaN,NaN,NaN,0.976286,None,NaN,...,NaN,NaN,NaN,NaN,None,None,None,None,NaN,NaN
Avg_IP_Number_of_Days_in_Hospital,None,NaN,NaN,NaN,NaN,NaN,0.999488,NaN,None,NaN,...,NaN,NaN,NaN,NaN,None,None,None,None,NaN,NaN
Avg_IP_Number_of_Days_in_Hospital.1,None,NaN,NaN,NaN,NaN,0.999488,NaN,NaN,None,NaN,...,NaN,NaN,NaN,NaN,None,None,None,None,NaN,NaN
Avg_IP_Number_of_Days_in_Hospital.2,None,NaN,NaN,0.827822,0.976286,NaN,NaN,NaN,None,NaN,...,NaN,NaN,NaN,NaN,None,None,None,None,NaN,NaN
Avg_IP_Number_of_Days_in_Hospital.3,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,...,NaN,NaN,NaN,NaN,None,None,None,None,NaN,NaN
OP_Claim_Count,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,...,0.955746,0.933761,0.935031,0.931964,None,None,None,None,NaN,NaN


In [26]:
corr_matrix.columns

Index(['PotentialFraud', 'IP_Claim_Count', 'IP_Benf_Count',
       'Avg_IP_InscClaimAmtReimbursed', 'Avg_IP_DeductibleAmtPaid',
       'Avg_IP_Number_of_Days_in_Hospital',
       'Avg_IP_Number_of_Days_in_Hospital.1',
       'Avg_IP_Number_of_Days_in_Hospital.2',
       'Avg_IP_Number_of_Days_in_Hospital.3', 'OP_Claim_Count',
       'OP_Benf_Count', 'Avg_OP_InscClaimAmtReimbursed',
       'Avg_OP_DeductibleAmtPaid', 'Avg_OP_Number_of_Days_in_Hospital',
       'Avg_OP_Number_of_Days_in_Hospital.1',
       'Avg_OP_Number_of_Days_in_Hospital.2', 'Total_Beneficiaries',
       'RenalDisease_Count', 'Alzheimer_Count', 'HeartFailure_Count',
       'KidneyDisease_Count', 'Cancer_Count', 'Pulmonary_Count',
       'Depression_Count', 'Diabetes_Count', 'IschemicHeart_Count',
       'Osteoporosis_Count', 'RheumatoidArthritis_Count', 'Stroke_Count',
       'Avg_PartA_Months', 'Avg_PartB_Months_Mode', 'Avg_IP_Reimbursement',
       'Avg_IP_Deductible', 'Avg_OP_Reimbursement', 'Avg_OP_Deductible'],
 

In [50]:
def correlated_vars(df):
    df1=df.copy()
    cols=list(df1.columns)    
    final_data=[]
    for index,row in df1.iterrows():
        cor_dict={}
        for c in cols:
            if 0.8<=abs(row[c])<1:
                k={c:row[c]}
                l=list(cor_dict.update(k))
            final_data.append(l)
    return final_data

correlated_vars(df=corr_matrix)

UnboundLocalError: local variable 'l' referenced before assignment

1.0
0.5253931054046992
0.5222557305392744
0.3075555268937365
0.31985536783085267
0.1562775929876254
0.15727106255761023
0.3320434678504162
0.20334057009913656
0.33580338914242047
0.3405502378836281
0.04676980519640057
0.041700915208835135
-0.005206264902285629
-0.05829491185949908
0.04232933850062358
0.3935310970429305
0.40996544241628047
0.38603451003479167
0.3764347485861789
0.3758755406956482
0.39158247973938504
0.381486495866275
0.38718987210256745
0.37589823294895736
0.36943256838382776
0.3905651831096477
0.38863453053950714
0.38968258905975695
0.009769833910861395
0.008369433632351186
0.1273981778197977
0.11989720065379376
-0.015203758422288082
-0.03154701516960908
0.5253931054046992
1.0
0.9974923038377357
0.32798513390016565
0.39876596457977936
0.22372848425358638
0.22422141546340923
0.41826532649444215
0.22800437445050895
0.20875848901088
0.2760373311623612
0.0493527092028107
0.0418273716126947
-0.003966718263774776
-0.059130301868437984
0.04963246650691962
0.39199230512963545
